## LDA Analysis

During our preliminary analysis, we discovered that relying on sklearn's native Perplexity metric to determine the optimal number of topics ($K$) created a severe disconnect from business reality. Because Perplexity strictly evaluates statistical predictive probability, it frequently leads to generated business topics that are completely incomprehensible to human reviewers.To prioritize the model's interpretability in real-world customer support scenarios, we decisively pivoted our technology stack. We adopted the Gensim library—which is specifically optimized for NLP tasks—and replaced Perplexity with the $C_v$ Coherence Score as our primary evaluation metric. This technical design choice ensured that the business pain points we ultimately extracted are not only mathematically cohesive, but also highly coherent from the perspective of human business intuition

This code snippet is an LDA analysis I wrote myself. It may differ from the final version and depends on the existing environment and data. It cannot run independently. If needed, please replace the corresponding part in the complete code and run the test.

In [ ]:
#The original Step 7 has been expanded to include a filter that targets only nouns.

# Step 7 - Lemmatization of tokens
lemmatizer = WordNetLemmatizer()
tag_map = {
    'J': nltk.corpus.wordnet.ADJ,
    'V': nltk.corpus.wordnet.VERB,
    'N': nltk.corpus.wordnet.NOUN,
    'R': nltk.corpus.wordnet.ADV
}

# Indiscriminate restoration for BoW / TF-IDF
def lemmatizing_tokens(tokens):
    tagged = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(
            token,
            tag_map.get(tag[0].upper(), nltk.corpus.wordnet.NOUN)
        )
        for token, tag in tagged
    ]

# Powerful filtration and reduction for LDA (Keep only the nouns)
def lemmatize_for_lda(tokens):
    tagged = pos_tag(tokens)
    final_tokens = []
    
    for token, tag in tagged:
        pos_prefix = tag[0].upper()
        if pos_prefix == 'N': 
            lemma = lemmatizer.lemmatize(token, tag_map.get(pos_prefix, nltk.corpus.wordnet.NOUN))
            final_tokens.append(lemma)
            
    return final_tokens

In [ ]:
# LDA Analysis
# Step1. Obtain minimal noun text strings dedicated to LDA.
lda_texts_series = preprocessing_pipeline(df, vectorization_method="lda", with_custom_stopword=True)

# Step2. Restore the string to token list.
# e.g., "password reset email" -> ['password', 'reset', 'email']
texts = [text.split() for text in lda_texts_series]

# Step3. Create a Gensim dictionary
id2word = corpora.Dictionary(texts)

# Step4. Filter extreme frequencies
# no_below=5: Remove terms that appear fewer than 5 times in the entire dataset
# no_above=0.85: Remove words that appear in more than 85% of the work orders
id2word.filter_extremes(no_below=5, no_above=0.85)

# Step5. Create a corpus to convert text into Gensim-specific bag-of-words vectors
corpus = [id2word.doc2bow(text) for text in texts]
print(f"\n Corpus created: Contains {len(id2word)} unique valid noun vocabulary items.")

# Step6. Define an evaluation function to find the optimal K value
def compute_coherence_values(dictionary, corpus, texts, start, limit, step):
    coherence_values = []
    model_list = []
    
    for num_topics in range(start, limit, step):
        print(f"Training the LDA model with K={num_topics} and calculating Coherence")
        
        model = LdaModel(
            corpus=corpus, 
            num_topics=num_topics, 
            id2word=dictionary, 
            passes=10, 
            random_state=42
        )
        model_list.append(model)
        
        # Calculate the C_v coherence score
        coherencemodel = CoherenceModel(
            model=model, 
            texts=texts, 
            dictionary=dictionary, 
            coherence='c_v'
        )
        coherence_values.append(coherencemodel.get_coherence())
        
    return model_list, coherence_values

# Step7. Run evaluation
start_k, limit_k, step_k = 2, 16, 1
print("\n Searching for the optimal number of topics")
model_list, coherence_values = compute_coherence_values(
    dictionary=id2word, 
    corpus=corpus, 
    texts=texts, 
    start=start_k, 
    limit=limit_k, 
    step=step_k
)

# Step8. Find the best K value model with the highest score
best_index = coherence_values.index(max(coherence_values))
best_k = range(start_k, limit_k, step_k)[best_index]
best_lda_model = model_list[best_index]
print(f"\n Optimal topic number K = {best_k} (Coherence Score = {coherence_values[best_index]:.4f})")

# Step9. Print the key terms of the best model's topics
print(f"\n Core Business Pain Points Extracted by the Best LDA Model (K={best_k})")
for topic_idx, topic_words in best_lda_model.print_topics(num_words=10):
    print(f" Topic {topic_idx}: {topic_words}")

# Step10. Visualize the Coherence line chart
plt.figure(figsize=(8, 5))
plt.plot(range(start_k, limit_k, step_k), coherence_values, marker='o', linestyle='-', color='purple', linewidth=2)
plt.scatter([best_k], [max(coherence_values)], color='red', zorder=5, s=100)
plt.annotate(
    f'Best K={best_k}',
    xy=(best_k, max(coherence_values)),
    xytext=(best_k + 0.3, max(coherence_values) - 0.02),
    arrowprops=dict(facecolor='black', arrowstyle='->'),
    fontsize=11
)
plt.title('LDA Coherence Score ($C_v$) vs Number of Topics', fontsize=14, fontweight='bold')
plt.xlabel("Number of Topics (K)", fontsize=12)
plt.ylabel("Coherence Score ($C_v$)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()